In [37]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [38]:
%pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


In [39]:
mnist_train = datasets.MNIST(
    root="./mnist_data", train=True, download=True, transform=transforms.ToTensor()
)
mnist_test = datasets.MNIST(
    root="./mnist_data", train=False, download=True, transform=transforms.ToTensor()
)

In [40]:
print(len(mnist_train), len(mnist_test))
print(mnist_train[0][0].shape, mnist_train[0][1])

60000 10000
torch.Size([1, 28, 28]) 5


In [41]:
batch_size = 32  # Hyperparameter
train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=False)

In [42]:
train_iter = iter(train_loader)
imgs, labels = next(train_iter)

print(imgs.shape, labels.shape)

torch.Size([32, 1, 28, 28]) torch.Size([32])


In [43]:
import torch
from torch import nn
from torch.nn import functional as F
from typing import List

class MLPClassifier(nn.Module):
    def __init__(self, input_dimension: int, hidden_layer_sizes: List[int], output_dimension: int):
        super(MLPClassifier, self).__init__()
        self.input_dimension = input_dimension
        self.hidden_layers = nn.ModuleList()
        previous_size = input_dimension
        for hidden_size in hidden_layer_sizes:
            self.hidden_layers.append(nn.Linear(previous_size, hidden_size))
            previous_size = hidden_size
        
        self.output_linear = nn.Linear(previous_size, output_dimension)

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        x = x.view(-1, self.input_dimension)

        for layer in self.hidden_layers:
            x = F.gelu(layer(x))


        output = self.output_linear(x)

        return output

In [ ]:
device = torch.device("cpu")
input_dimension = 1 * 28 * 28  # MNIST images are 28x28 pixels
output_dimension = 10  # Number of classes in MNIST
hidden_layer_sizes = [128, 128, 128]  # Example hidden layer sizes
num_epochs = 5
learning_rate = 0.01

model = MLPClassifier(input_dimension, hidden_layer_sizes, output_dimension)
model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
lr_decay_rate = 0.8
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=lr_decay_rate)

In [45]:
output = model(imgs)

In [47]:
from typing import Tuple


def evaluate(
    model: nn.Module,
    test_loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> Tuple[float, float]:

    loss = 0.0
    accuracy = 0.0
    total = 0

    with torch.no_grad():
        for input_images, labels in test_loader:
            input_images = input_images.to(device)
            labels = labels.to(device)
            

            outputs = model(input_images)

            # (batch_size, num_classes)
            predicted = torch.argmax(outputs, 1)


            batch_size = labels.size(0)
            total += batch_size
            

            accuracy += (predicted == labels).sum().item()
            loss += criterion(outputs, labels).item() * batch_size

    accuracy /= total
    loss /= total

    return accuracy, loss

In [48]:
evaluate(model, test_loader, criterion, device)

(0.1009, 2.3047402770996093)

In [55]:
test_accuracy, test_loss = evaluate(model, test_loader, criterion, device)
print(f"[test] Initial - accuracy: {test_accuracy:.4f}, loss: {test_loss:.4f}")

for epoch in range(num_epochs):  # Number of epochs
    train_accuracy = 0.0
    train_loss = 0.0
    
    model.train()
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        optimizer.step()
        optimizer.zero_grad()
        
        batch_size = inputs.size(0)
        train_loss += loss.item() * batch_size
        predicted = torch.argmax(outputs, dim = 1)
        train_accuracy += (predicted == labels).sum().item()
        
    lr_scheduler.step()
    
    train_accuracy /= len(train_loader.dataset)
    train_loss /= len(train_loader.dataset)
    
    model.eval()
    test_accuracy,test_loss = evaluate(model, test_loader, criterion, device)
    
    print(f"[Train] {epoch} / {num_epochs} - accuracy: {train_accuracy:.4f}, loss: {train_loss:.4f}")
    print(f"[test] {epoch} / {num_epochs} - accuracy: {test_accuracy:.4f}, loss: {test_loss:.4f}")

[test] Initial - accuracy: 0.0591, loss: 2.3029
[Train] 0 / 30 - accuracy: 0.9074, loss: 0.3343
[test] 0 / 30 - accuracy: 0.9432, loss: 0.1996
[Train] 1 / 30 - accuracy: 0.9512, loss: 0.1824
[test] 1 / 30 - accuracy: 0.9553, loss: 0.1697
[Train] 2 / 30 - accuracy: 0.9635, loss: 0.1367
[test] 2 / 30 - accuracy: 0.9595, loss: 0.1457
[Train] 3 / 30 - accuracy: 0.9711, loss: 0.1086
[test] 3 / 30 - accuracy: 0.9670, loss: 0.1238
[Train] 4 / 30 - accuracy: 0.9765, loss: 0.0842
[test] 4 / 30 - accuracy: 0.9706, loss: 0.1064
[Train] 5 / 30 - accuracy: 0.9812, loss: 0.0653
[test] 5 / 30 - accuracy: 0.9754, loss: 0.0981
[Train] 6 / 30 - accuracy: 0.9852, loss: 0.0511
[test] 6 / 30 - accuracy: 0.9702, loss: 0.1062
[Train] 7 / 30 - accuracy: 0.9886, loss: 0.0384
[test] 7 / 30 - accuracy: 0.9741, loss: 0.1002
[Train] 8 / 30 - accuracy: 0.9917, loss: 0.0275
[test] 8 / 30 - accuracy: 0.9764, loss: 0.0939
[Train] 9 / 30 - accuracy: 0.9938, loss: 0.0204
[test] 9 / 30 - accuracy: 0.9761, loss: 0.1024
[T